# Spark Session, schema configuration and csv data loading

## Verifying the CWD

In [0]:
import os
print(os.getcwd())

## Initiating the sparksession

In [0]:
from pyspark.sql import SparkSession

# Initiate spark session
spark = (
        SparkSession.
        builder.
        appName("SparkSession").
        getOrCreate()
)

## Defining an explicit schema for the data

I'm using [this dataset](https://www.kaggle.com/datasets/uradkr/alabama-sold-real-estate-intelligence-2026) for my analysis

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, FloatType, BooleanType, DateType

alabama_real_estate_schema = StructType([
    StructField("type", StringType(), True),
    StructField("sub_type", StringType(), True),
    StructField("listPrice", FloatType(), True),
    StructField("lastSoldPrice", FloatType(), True),
    StructField("soldOn", StringType(), True),  # Read as string to handle date format
    StructField("sqft", FloatType(), True),
    StructField("stories", FloatType(), True),
    StructField("beds", FloatType(), True),
    StructField("baths", FloatType(), True),
    StructField("baths_full", FloatType(), True),
    StructField("baths_full_calc", FloatType(), True),
    StructField("garage", FloatType(), True),
    StructField("year_built", FloatType(), True),
    StructField("postal_code", StringType(), True),
    StructField("is_valid_al_zip", BooleanType(), True),
    StructField("sold_year", FloatType(), True),
    StructField("sold_month", FloatType(), True),
    StructField("sold_quarter", FloatType(), True),
    StructField("property_age", FloatType(), True),
    StructField("sold_to_list_ratio", FloatType(), True),
    StructField("price_premium_pct", FloatType(), True),
    StructField("negotiation_outcome", StringType(), True),
    StructField("ratio_outlier_flag", BooleanType(), True),
    StructField("price_per_sqft", FloatType(), True),
    StructField("text_clean", StringType(), True),
    StructField("pii_redacted_flag", BooleanType(), True)
])

columns: list[str] = alabama_real_estate_schema.fieldNames() # Retrieve columns


In [0]:
print(columns)

## Reading the data

In [0]:
login:str = input("Please enter your login")

absolute_data_path = f"""/Workspace/Users/{login}@softserve.academy/lab1 - Databricks Fundamentals & DEV Setup/lab1/assignment/alabama_sold_real_estate_intelligence_2026.csv"""

## Making sure the provided absolute path does exist

In [0]:
# We're checking if the data path provided by the user exists or not - defensive programming

from pyspark.sql.functions import try_to_date

try:
    dbutils.fs.ls(absolute_data_path)
    print(f"Your path '{absolute_data_path}' exists. Reading the data from the csv...")
    
    alabama_df = spark.read.csv(absolute_data_path,
               schema=alabama_real_estate_schema, 
               header=True,
               ).select(columns)
    
    # After loading the DataFrame, convert soldOn to YYYY-MM-DD format
    alabama_df = alabama_df.withColumn("soldOn", try_to_date("soldOn", "yyyy-MM-dd"))

    alabama_df.printSchema()
                 
except Exception as e:
    raise FileNotFoundError(f"Your path {absolute_data_path} doesn't exist!")

# Loading the data

## Creating the schema

In [0]:
catalog = spark.catalog.currentCatalog()
schema = f"{catalog}.{login}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

## Specifying the path of the sourced table

In [0]:
table_name = "alabama_real_estate"
full_path = f"{schema}.{table_name}"

In [0]:
(alabama_df.write.
 format("delta").mode("overwrite").option("overwriteSchema", "true").
 saveAsTable(full_path)
 )

# Simple spark transformations (grouping, sorting, filtering etc.)

In [0]:
alabama_df.show(3)

## Renaming the columns for clarity

In [0]:
# Rename columns to shorter, relevant names
short_names = {
    "property_id": "pid",
    "address": "addr",
    "city": "cty",
    "state": "st",
    "zip_code": "zip",
    "listing_price": "price",
    "bedrooms": "beds",
    "bathrooms": "baths",
    "square_feet": "sqft",
    "listing_date": "list_date",
    "year_built": "year_built",
    "property_type": "prop_type",
    "lot_size": "lotsz",
    "hoa_fee": "hoa",
    "description": "desc",
    "agent_name": "agent",
    "agent_phone": "agent_phone",
    "status": "stat"
}

for old, new in short_names.items():
    alabama_df = alabama_df.withColumnRenamed(old, new)
display(alabama_df)

## Counting missing values across each column

In [0]:
from pyspark.sql.functions import when, isnull, count

null_count = alabama_df.select(*[count(when(isnull(c), c)).alias(c) for c in columns])
null_count.show(1)

## Grouping and sorting

In [0]:
from pyspark.sql.functions import avg, min, max, count, round

# Let's only select interesting columns
df_subset = alabama_df.select("sub_type", "baths", "beds")

# Group by subtypes and take min, max and average of number of baths and beds
grouped_df = df_subset.groupby("sub_type").agg(
    count("*").alias("count"),
    min("baths").alias("min_baths"),
    max("baths").alias("max_baths"),
    round(avg("baths"), 1).alias("avg_baths"),
    min("beds").alias("min_beds"),
    max("beds").alias("max_beds"),
    round(avg("beds"), 1).alias("avg_beds")
)

display(grouped_df) # Why some houses don't have any baths in there? 

In [0]:
no_bath_props = alabama_df.filter(col("baths")==0).select("type", "sub_type", "baths")
no_bath_prop_type_count = no_bath_props.groupby("sub_type").agg(count("*"))

In [0]:
display(no_bath_prop_type_count) # Some houses don't have baths or maybe the issue is data quality.

In [0]:
from pyspark.sql.functions import desc, col

grouped_sorted_df = grouped_df.sort(desc(col("count")))
display(grouped_sorted_df) # Single family houses are the most common property type in Alabama followed up by condos